# Aedes-AI GRU Model with Mean Temperature as Input


This code trains and tests a GRU model that estimates Aedes aegypti abundance from the following time series.

* Daily mean temperature (C)
* Daily precipitation (cm)
* Daily relative humidity

The GRU is trained to reproduce the results of MoLS.

## Create new configuration file


A new configuration file for the ANN model needs to be created. In particular, since there is only one temperature time series, the samples have size 90x3. This cell only needs to be run once.

In [1]:
import os, json

In [ ]:
fpath='../utils/model_files/gru_avg_temp_config.json'

with open(fpath, 'r+') as fp:
  data = json.load(fp)
  data['data']['data_shape']=[90,3]
  data['files']['model']='../utils/model_files/gru_avg_temp.h5'
  data['files']['training']='../utils/model_files/train_avg_data.pd'
  data['files']['validation']='../utils/model_files/val_avg_data.pd'
  data['files']['testing']='../utils/model_files/test_avg_data.pd'
with open(fpath, 'w') as fp:
  json.dump(data,fp)

## Train the new GRU model

We are now ready to train the model. Additional definitions are imported from `training.py` in the utils directory.

The resulting model will be added to the local utils/model_files directory.


In [140]:
sys.path.append( os.path.abspath(os.path.join('..')) )
import sys
import tensorflow as tf
import tensorflow.keras.backend as K
import utils.models as models
import utils.training as training_utils

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
import pickle
import pdb

import utils.predictions as predictions

In [ ]:
#os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

gpus = tf.config.list_physical_devices('GPU')
if gpus:
  # Restrict TensorFlow to only allocate 3GB of memory on the first GPU
  try:
    tf.config.experimental.set_virtual_device_configuration(
        gpus[0],
        [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=2*1024)])
    logical_gpus = tf.config.experimental.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    # Virtual devices must be set before GPUs have been initialized
    print(e)

np.random.seed(14)

# load the model as necessary
with open("../utils/model_files/gru_avg_temp_config.json") as fp:
  config=json.load(fp)

model_file = os.path.expanduser(config['files']['model'])
if os.path.exists(model_file):
  model = tf.keras.models.load_model(model_file, custom_objects = {'r2_keras': predictions.r2_keras})
  print(model_file + ' already exists')
else:
  model = getattr(models, config['model'])(config['data']['data_shape'])
     
  summer_cities = set(pd.read_csv(os.path.expanduser('../utils/model_files/hi_locs.csv'), squeeze=True, index_col=0))
  winter_cities = {"Dane,Wisconsin", "Milwaukee,Wisconsin", "New Haven,Connecticut", "Bronx,New York", "Kings,New York",
                     "Monmouth,New Jersey", "Mono,California", "Monterey,California", "Morris,New Jersey", "Napa,California",
                     "Nassau,New York", "New Hanover,North Carolina", "New River,Arizona", "Okaloosa,Florida", "Orange,California",
                     "Oro Valley,Arizona", "Prescott,Arizona", "Rio Rico,Arizona", "Rockland,New York", "Sacramento,California"}

  # get the data
  training = pd.read_pickle(os.path.expanduser(config['files']['training']))
  validation = pd.read_pickle(os.path.expanduser(config['files']['validation']))
  testing = pd.read_pickle(os.path.expanduser(config['files']['testing']))
  training, scaler = training_utils.format_data(training, config['data']['data_shape'], config['data']['samples_per_city'],
                                   scaler=MinMaxScaler(), fit_scaler=True,
                                   summer_samples=config['data']['summer_samples'],
                                   winter_samples=config['data']['winter_samples'],
                                   summer_cities=summer_cities,
                                   winter_cities=winter_cities)
  with open('../utils/model_files/avg_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
    
  validation = training_utils.format_data(validation, config['data']['data_shape'], config['data']['samples_per_city'],
                             scaler=scaler)
  testing = training_utils.format_data(testing, config['data']['data_shape'], config['data']['samples_per_city'],
                          scaler=scaler)
   
  X_train, y_train = training_utils.split_and_shuffle(training)
  X_val, y_val = training_utils.split_and_shuffle(validation)
  X_test, y_test = training_utils.split_and_shuffle(testing)

  model.compile(optimizer = getattr(tf.keras.optimizers, config['compile']['optimizer'])(lr = config['compile']['learning_rate']),
                      loss = config['compile']['loss'], metrics = [predictions.r2_keras])
  history = model.fit(X_train, y_train, validation_data = (X_val, y_val), **config['fit'],
                  callbacks = [tf.keras.callbacks.TensorBoard(), tf.keras.callbacks.EarlyStopping(patience = 15, restore_best_weights = True)])
  model.save(model_file, save_format = 'h5')

In [142]:
import importlib
importlib.reload(predictions)
#model = tf.keras.models.load_model('../utils/model_files/gru_avg_temp.h5', custom_objects = {'r2_keras': predictions.r2_keras})

with open ('../utils/model_files/avg_scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)
    
data_shape = config['data']['data_shape']
test_data = pd.read_pickle(config['files']['testing'])

results = predictions.gen_preds(model, test_data, data_shape, loaded_scaler, fit_scaler=False)